### Online Retail II — Data Cleaning

Cleans the raw combined dataset (`data/online_retail_ii.parquet`) into a modeling-ready file.

must:
1. Load + profile (this notebook)
2. Drop rows with missing Customer ID
3. Decide on cancellations / returns (negative Quantity, invoices starting with "C")
4. Handle bad Price rows (<= 0)
5. Deduplicate
6. Handle outliers in Quantity / Price
7. Build customer-level activity timeline
8. Apply the 90-day rule
9. Re-profile and save cleaned output

## 1. Load + profile

In [1]:
# load the raw combined dataset & confirm it read in 
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_parquet('../data/online_retail_ii.parquet')
df.shape

(1067371, 8)

In [2]:
# column formats
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# confirm types are what we expect 
df.dtypes

Invoice                string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                string
dtype: object

In [4]:
# overall missing-value scan across all columns
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [5]:
# summary stats across numeric & categorical columns
df.describe(include='all')

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371,1067371,1062989,1.067371e+06,1067371,1.067371e+06,824364.000000,1067371
unique,53628,5305,5698,NaN,NaN,NaN,NaN,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1350,5829,5918,NaN,NaN,NaN,NaN,981330
mean,NaN,NaN,NaN,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504,NaN
min,NaN,NaN,NaN,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000,NaN
50%,NaN,NaN,NaN,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000,NaN
75%,NaN,NaN,NaN,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000,NaN
max,NaN,NaN,NaN,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000,NaN


In [6]:
# size the missing-Customer-ID problem before deciding whether it's safe to drop those rows
missing_customer_id = df['Customer ID'].isna().sum()
print('Missing Customer ID rows:', missing_customer_id, f'({missing_customer_id / len(df):.1%} of rows)')

Missing Customer ID rows: 243007 (22.8% of rows)


## 2. Drop rows with missing Customer ID

Almost every downstream step (customer-month table, churn labels) needs a valid Customer ID, so this comes first

In [7]:
# drop rows with missing Customer ID
df.dropna(subset=['Customer ID'], inplace=True)

# turn Customer ID column into int type
df['Customer ID'] = df['Customer ID'].astype(int)

# check the changes
print(df.shape)
print(df['Customer ID'].dtype)

(824364, 8)
int64


## 3. Bad Price rows (<= 0)

TODO: inspect a sample, then drop

In [8]:
# drop rows where Price <= 0
df.drop(index=df[df['Price'] <= 0].index, inplace=True)

## 4. Deduplicate

TODO: decide whether exact duplicate rows are true dupes or legitimate repeated line items

In [9]:
# drop duplicate rows
df.drop_duplicates(inplace=True)

## 5. Handle missing values in description

TODO: Replace missing description values with "Unknown"

In [10]:
# Replace missing description with "Unknown"
df['Description'] = df['Description'].fillna('Unknown')

## 6. Handle non-product administrative codes

TODO: drop non-product rows

In [11]:
# Strip whitespace
df['StockCode'] = df['StockCode'].astype(str).str.strip()

# Matches standard 5-digit product codes with any letter suffix (e.g., '85048', '15056BL')
is_standard_product = df['StockCode'].str.match(r'^\d{5}[A-Za-z]*$')

# Genuine products that do not use the standard 5-digit convention
valid_nonstandard_products = {'PADS', 'SP1002'}

# Flag rows that fail the format AND are not valid exceptions
is_non_product = (~is_standard_product) & (~df['StockCode'].isin(valid_nonstandard_products))

# Inspect dropped rows
print(f"Dropping {is_non_product.sum()} non-product rows ({is_non_product.mean():.2%})")
print(df[is_non_product]['StockCode'].value_counts().head(10))

# Filter dataset
df = df[~is_non_product].copy()

Dropping 3632 non-product rows (0.46%)
StockCode
POST            1983
M               1078
C2               254
D                170
ADJUST            61
BANK CHARGES      37
DOT               16
CRUK              16
TEST001           13
ADJUST2            3
Name: count, dtype: int64


## 7. Cancellations / returns

TODO: decide keep vs. remove vs. keep-as-separate-signal for invoices starting with "C" / negative Quantity

In [12]:
# Flag cancellation rows (Invoice starting with 'C' or negative Quantity)
df['Is_Cancellation'] = df['Invoice'].astype(str).str.startswith('C') | (df['Quantity'] < 0)

# Ensure InvoiceDate is datetime for chronological cumulative calculation
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Sort chronologically by customer to accurately cumulate transactions
df.sort_values(by=['Customer ID', 'InvoiceDate'], inplace=True)

# Cumulative sum of transaction quantities (including negative cancellation quantities)
df['Net_Quantity'] = df.groupby('Customer ID')['Quantity'].cumsum()

print(f"Cancellations tracked: {(df['Quantity'] < 0).sum()} rows")
print(df[['Customer ID', 'InvoiceDate', 'Quantity', 'Net_Quantity']].head())

# Drop cancellations directly by invoice prefix and ensure positive quantity
# df = df[~df['Invoice'].astype(str).str.startswith('C')].copy()
# df = df[df['Quantity'] > 0]

Cancellations tracked: 17587 rows
        Customer ID         InvoiceDate  Quantity  Net_Quantity
107800        12346 2010-03-02 13:08:00         1             1
107801        12346 2010-03-02 13:08:00         1             2
107802        12346 2010-03-02 13:08:00         1             3
107803        12346 2010-03-02 13:08:00         1             4
107804        12346 2010-03-02 13:08:00         1             5


## 8. Outliers in Quantity / Price

TODO: pick a method (IQR / percentile capping / z-score); consider per-customer bulk orders before capping blindly

In [13]:
# Define percentile cutoffs using valid positive purchases
pos_qty_mask = df['Quantity'] > 0
q_cap = df.loc[pos_qty_mask, 'Quantity'].quantile(0.99)
p_cap = df['Price'].quantile(0.99)

print(f"Quantity 99th percentile cutoff: {q_cap}")
print(f"Price 99th percentile cutoff: {p_cap:.2f}")

# Winsorize Price (upper tail only, since Price > 0)
df['Price'] = df['Price'].clip(upper=p_cap)

# Winsorize Quantity symmetrically:
# Upper-cap purchases at +q_cap, lower-cap cancellations at -q_cap
df['Quantity'] = df['Quantity'].clip(lower=-q_cap, upper=q_cap)

# Recompute Total_Price with winsorized values
df['Total_Price'] = df['Quantity'] * df['Price']

# Recompute Net_Quantity after clipping outlier transaction sizes
df['Net_Quantity'] = df.groupby('Customer ID')['Quantity'].cumsum()

# Verify distribution changes
print(df[['Quantity', 'Price', 'Total_Price', 'Net_Quantity']].describe())

Quantity 99th percentile cutoff: 144.0
Price 99th percentile cutoff: 12.75
            Quantity          Price    Total_Price   Net_Quantity
count  794183.000000  794183.000000  794183.000000  794183.000000
mean       10.749808       2.860482      18.758992    5127.064998
std        20.113398       2.677393      40.104479   17107.504670
min      -144.000000       0.001000   -1576.800000   -2408.000000
25%         2.000000       1.250000       4.350000     307.000000
50%         5.000000       1.950000      11.700000     980.000000
75%        12.000000       3.750000      19.500000    2920.000000
max       144.000000      12.750000    1836.000000  286186.000000


## 9. Customer-level activity timeline

TODO: build Customer ID -> list of InvoiceDates (prerequisite for the 90-day rule)

In [14]:
# Map Custumer ID -> list of unique InvoiceDates
customer_timeline = (
    df[['Customer ID', 'InvoiceDate']] # Select relevant columns
    .drop_duplicates() # Keep unqiue visits
    .sort_values(['Customer ID', 'InvoiceDate']) # Sort chronologically
    .groupby('Customer ID')['InvoiceDate'] # Group dates by customer
    .apply(list) # Convert dates to a list
)

# Sanity checks
print(f"Total unique customers: {len(customer_timeline)}")
print("\nSample customer timeline:")
sample_id = customer_timeline.index[0]
print(f"Customer ID {sample_id} ({len(customer_timeline[sample_id])} purchase dates):")
print(customer_timeline[sample_id][:5])

Total unique customers: 5875

Sample customer timeline:
Customer ID 12346 (4 purchase dates):
[Timestamp('2010-03-02 13:08:00'), Timestamp('2010-06-28 13:53:00'), Timestamp('2011-01-18 10:01:00'), Timestamp('2011-01-18 10:17:00')]


## 10. 90-day rule

Customer-months too close to the dataset end date (2011-12-09) can't be labeled, because their full 90-day forward window isn't observed.

We do NOT delete those transactions. They are the forward window used to label earlier months (an August customer is labeled using Sep-Nov purchases).
This step only tags each row with its month and finds which months are unlabelable. The cutoff is applied to the snapshots at labeling time.

In [15]:
# Last date in the data, and the last date that still has 90 days of data after it
dataset_end_date = pd.Timestamp('2011-12-09')
cutoff_date = dataset_end_date - pd.Timedelta(days=90)

# Make sure InvoiceDate is a datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Tag each row with its customer-month (e.g. 2010-03)
df['CustomerMonth'] = df['InvoiceDate'].dt.to_period('M')

# Flag months whose month-end is after the cutoff (these can't be labeled)
# We only count them here; no rows are dropped
unlabelable = df['CustomerMonth'].dt.to_timestamp('M') > cutoff_date

print(f"90-day cutoff date: {cutoff_date.date()}")
print(f"Rows kept: {len(df):,}")
print(f"Rows in unlabelable months (kept as forward window): {unlabelable.sum():,}")
print(f"Latest customer-month in data: {df['CustomerMonth'].max()}")

90-day cutoff date: 2011-09-10
Rows kept: 794,183
Rows in unlabelable months (kept as forward window): 171,221
Latest customer-month in data: 2011-12


## 11. Re-profile and save

In [16]:
# Re-profile the cleaned dataset

print("Final dataset shape:", df.shape)

# Column types
print("\nColumn types:")
print(df.dtypes)

# Missing values
print("\nMissing values:")
print(df.isna().sum())

# Summary statistics
print("\nSummary statistics:")
print(df.describe(include='all'))

# Date range
print("\nInvoice date range:")
print(df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())

# Check for remaining cancellations
print("\nRemaining cancellations:", 
      df['Invoice'].astype(str).str.startswith('C').sum())

# Check for non-positive values
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("Price <= 0:", (df['Price'] <= 0).sum())

# Drop duplicate rows again
df.drop_duplicates(inplace=True)

# Check duplicates
print("Duplicate rows:", df.duplicated().sum())

# Check df.head()
print("\n\n", df.head())

Final dataset shape: (794183, 12)

Column types:
Invoice                    string
StockCode                     str
Description                string
Quantity                    int64
InvoiceDate        datetime64[ns]
Price                     float64
Customer ID                 int64
Country                    string
Is_Cancellation              bool
Net_Quantity                int64
Total_Price               float64
CustomerMonth           period[M]
dtype: object

Missing values:
Invoice            0
StockCode          0
Description        0
Quantity           0
InvoiceDate        0
Price              0
Customer ID        0
Country            0
Is_Cancellation    0
Net_Quantity       0
Total_Price        0
CustomerMonth      0
dtype: int64

Summary statistics:
       Invoice StockCode                         Description       Quantity  \
count   794183    794183                              794183  794183.000000   
unique   43877      4635                                5286        

In [17]:
# Save cleaned dataset (full date range, through 2011-12-09) to the data directory
df.to_parquet('../data/online_retail_ii_cleaned_v3.parquet', index=False)

print("Cleaned dataset saved to data/online_retail_ii_cleaned_v3.parquet")

Cleaned dataset saved to data/online_retail_ii_cleaned_v3.parquet
